# 02 · Análise exploratória

Entende o snapshot da Gold antes de modelar e registra hipóteses ligadas a decisões de modelagem (requisitos FR2.1). Os gráficos vêm de `src/visualization/graficos.py` e são gravados em `images/`.

Dois alvos: **T1**, o aluno (`alfabetizado`), e **T2**, o município (`atingiu_meta`, só 2024). A classe de risco é *não alfabetizado* e *não atingiu a meta*.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from sklearn.metrics import roc_auc_score

from src.preprocessing.loader import carregar_tabela
from src.preprocessing.qualidade import descrever_colunas
from src.visualization.graficos import barras_proporcao_alvo, histograma, mapa_correlacao

alunos = carregar_tabela('fact_alunos')
municipios = carregar_tabela('dim_municipio')
metas = carregar_tabela('fact_meta_resultado_municipio')
redes = carregar_tabela('dim_rede')
print(alunos.shape, municipios.shape, metas.shape)

(3867999, 11) (5571, 9) (10698, 11)


## 1. Alvo do aluno (T1)

Proporção de alfabetizados por ano, separando quem preencheu o caderno de prova de quem não preencheu.

In [2]:
alunos['prova_feita'] = alunos['preenchimento_caderno']
resumo = alunos.groupby(['ano', 'prova_feita'])['alfabetizado'].agg(alunos='count', alfabetizados='sum', proporcao='mean')
resumo['proporcao'] = resumo['proporcao'].round(3)
resumo

alunos  alfabetizados  proporcao
ano  prova_feita                                   
2023 False         244630              0      0.000
     True         1502809         877427      0.584
2024 False         268708              0      0.000
     True         1851852        1107119      0.598

Quem não preencheu o caderno (13,3% dos alunos, `proficiencia` nula) é sempre `alfabetizado` falso. Misturar esse grupo no alvo mede ausência, não leitura. A população do modelo do aluno precisa ser decidida na unidade de pré-processamento.

In [3]:
com_prova = alunos[alunos['prova_feita']]
barras_proporcao_alvo(com_prova, 'ano', 'alfabetizado', 'Alfabetizados por ano (quem fez a prova)', 'eda_alfabetizado_por_ano.png')
histograma(com_prova, 'proficiencia', 'Proficiência (quem fez a prova)', 'eda_proficiencia.png')
com_prova['proficiencia'].describe().round(1)

count    3354661.0
mean         748.4
std           47.6
min          578.5
25%          719.8
50%          753.2
75%          779.6
max          904.4
Name: proficiencia, dtype: float64

A proficiência é contínua e o corte de 743 define `alfabetizado` (verificado nos 3.354.661 alunos com proficiência).

## 2. Alvo do município (T2)

Só 2024 tem rótulo. O rótulo é nulo quando não há meta vigente.

In [4]:
m2024 = metas[metas['ano'] == 2024].merge(municipios, on='id_municipio', how='left')
print('linhas 2024:', len(m2024), '| sem rótulo:', m2024['atingiu_meta'].isna().sum())
print(m2024['atingiu_meta'].value_counts().rename({True: 'atingiu', False: 'não atingiu'}))
print(m2024.groupby('rede')['atingiu_meta'].agg(['count', 'mean']).round(3))

linhas 2024: 5346 | sem rótulo: 120
atingiu_meta
atingiu        2782
não atingiu    2444
Name: count, dtype: int64
           count      mean
rede                      
Municipal   5226  0.532338


Todos os rótulos de 2024 são da rede municipal. A classe positiva de T2 (não atingiu a meta) tem 2.444 casos, cerca de 47%.

## 3. Valores ausentes

In [5]:
tabelas = {'fact_alunos': alunos.drop(columns=['prova_feita']), 'fact_meta_resultado_municipio': metas, 'dim_municipio': municipios,
           'fact_indicador_municipio': carregar_tabela('fact_indicador_municipio')}
ausentes = pd.concat([descrever_colunas(t, d) for t, d in tabelas.items()])
ausentes[ausentes['pct_ausentes'] > 0][['tabela', 'coluna', 'ausentes', 'pct_ausentes']].round(2).reset_index(drop=True)

,tabela,coluna,ausentes,pct_ausentes
0,fact_alunos,proficiencia,513338,13.27
1,fact_alunos,peso_aluno,513338,13.27
2,fact_meta_resultado_municipio,taxa_alfabetizacao,120,1.12
3,fact_meta_resultado_municipio,meta_indicador,5472,51.15
4,fact_meta_resultado_municipio,gap_pontos,5472,51.15
5,fact_meta_resultado_municipio,atingiu_meta,5472,51.15
6,fact_meta_resultado_municipio,percentual_participacao,120,1.12
7,fact_meta_resultado_municipio,valid_to,5352,50.03
8,dim_municipio,capital_uf,1,0.02
9,dim_municipio,idhm,6,0.11


`proficiencia` e `peso_aluno` faltam para quem não fez a prova. `meta_indicador` falta em 2023 (sem meta vigente). As `proporcao_aluno_nivel_*` faltam em cerca de 48% das linhas. Em `dim_municipio`, 6 de 5.571 municípios não têm IDHM, o que exige imputação numérica.

## 4. Alvo por região, faixa de IDHM, UF e rede

Trilha T1, só quem fez a prova.

In [6]:
t1 = com_prova.merge(municipios[['id_municipio', 'nome_regiao', 'sigla_uf', 'idhm']], on='id_municipio', how='left')
t1['faixa_idhm'] = pd.qcut(t1['idhm'], 5, duplicates='drop').astype(str)
barras_proporcao_alvo(t1, 'nome_regiao', 'alfabetizado', 'Alfabetizados por região', 'eda_alfabetizado_por_regiao.png')
barras_proporcao_alvo(t1, 'faixa_idhm', 'alfabetizado', 'Alfabetizados por quintil de IDHM', 'eda_alfabetizado_por_faixa_idhm.png')
barras_proporcao_alvo(t1, 'sigla_uf', 'alfabetizado', 'Alfabetizados por UF', 'eda_alfabetizado_por_uf.png')
print(t1.groupby('nome_regiao')['alfabetizado'].agg(['count', 'mean']).round(3))
print(t1.groupby('faixa_idhm')['alfabetizado'].agg(['count', 'mean']).round(3))
print(t1.merge(redes, on='rede').groupby('rede_desc')['alfabetizado'].agg(['count', 'mean']).round(3))

                count   mean
nome_regiao                 
Centro-Oeste   332882  0.620
Nordeste       968531  0.558
Norte          368708  0.509
Sudeste       1107436  0.613
Sul            577104  0.645
                 count   mean
faixa_idhm                   
(0.417, 0.629]  671680  0.551
(0.629, 0.7]    676192  0.594
(0.7, 0.737]    684144  0.598
(0.737, 0.772]  655696  0.607
(0.772, 0.862]  665649  0.608
             count   mean
rede_desc                
Estadual    372596  0.621
Municipal  2982041  0.588
Privada         24  0.667


Há gradiente por região (Norte 50,9% a Sul 64,5%) e por IDHM (55,1% no quintil mais baixo a 60,8% no mais alto), mas ele é modesto. A rede é quase constante: 24 alunos de rede privada entre 3,35 milhões.

Trilha T2: proporção que atingiu a meta em 2024, por região.

In [7]:
t2 = m2024.dropna(subset=['atingiu_meta']).copy()
t2['atingiu_meta'] = t2['atingiu_meta'].astype(int)
barras_proporcao_alvo(t2, 'nome_regiao', 'atingiu_meta', 'Municípios que atingiram a meta em 2024, por região', 'eda_atingiu_meta_por_regiao.png')
t2.groupby('nome_regiao')['atingiu_meta'].agg(['count', 'mean']).round(3)

,count,mean
nome_regiao,,
Centro-Oeste,460,0.730
Nordeste,1735,0.500
Norte,389,0.437
Sudeste,1589,0.662
Sul,1053,0.338


O Sul, que tem a maior proporção de alfabetizados, tem a **menor** de municípios que atingiram a meta (33,8%). A meta de cada município parece crescer com o ponto de partida; a seção 6 testa isso.

## 5. Correlações

Pearson e Spearman no nível município e rede: IDHM, taxas de 2023 e 2024 e meta de 2024.

In [8]:
p = metas.pivot_table(index=['id_municipio', 'rede'], columns='ano', values=['taxa_alfabetizacao', 'meta_indicador'], aggfunc='last')
p.columns = [f'{a}_{b}' for a, b in p.columns]
p = p.reset_index().merge(municipios, on='id_municipio', how='left')
colunas = ['idhm', 'idhm_educacao', 'idhm_renda', 'idhm_longevidade', 'capital_uf', 'taxa_alfabetizacao_2023', 'taxa_alfabetizacao_2024', 'meta_indicador_2024']
mapa_correlacao(p, colunas, 'pearson', 'Correlação de Pearson', 'eda_correlacao_pearson.png')
mapa_correlacao(p, colunas, 'spearman', 'Correlação de Spearman', 'eda_correlacao_spearman.png')
p[colunas].corr('pearson').round(2)

,idhm,idhm_educacao,idhm_renda,idhm_longevidade,capital_uf,taxa_alfabetizacao_2023,taxa_alfabetizacao_2024,meta_indicador_2024
idhm,1.00,0.95,0.95,0.85,0.10,0.29,0.23,0.33
idhm_educacao,0.95,1.00,0.82,0.71,0.10,0.27,0.23,0.30
idhm_renda,0.95,0.82,1.00,0.84,0.11,0.27,0.18,0.32
idhm_longevidade,0.85,0.71,0.84,1.00,0.04,0.27,0.22,0.32
capital_uf,0.10,0.10,0.11,0.04,1.00,-0.03,-0.03,-0.02
taxa_alfabetizacao_2023,0.29,0.27,0.27,0.27,-0.03,1.00,0.64,0.98
taxa_alfabetizacao_2024,0.23,0.23,0.18,0.22,-0.03,0.64,1.00,0.63
meta_indicador_2024,0.33,0.30,0.32,0.32,-0.02,0.98,0.63,1.00


O IDHM geral e seus três componentes são fortemente correlacionados (0,71 a 0,95), então carregam informação redundante. A taxa de 2024 se associa pouco ao IDHM (0,23). A meta de 2024 é quase função da taxa de 2023.

## 6. Sinal para prever a meta de 2024 com dados anteriores (T2)

In [9]:
q = p.merge(t2[['id_municipio', 'rede', 'atingiu_meta']], on=['id_municipio', 'rede'], how='inner')
q['folga_2023'] = q['taxa_alfabetizacao_2023'] - q['meta_indicador_2024']
q = q.dropna(subset=['folga_2023', 'idhm'])
print('linhas:', len(q))
print('corr(meta 2024, taxa 2023) =', round(q['meta_indicador_2024'].corr(q['taxa_alfabetizacao_2023']), 3))
print('corr(taxa 2024, taxa 2023) =', round(q['taxa_alfabetizacao_2024'].corr(q['taxa_alfabetizacao_2023']), 3))
for coluna in ['folga_2023', 'taxa_alfabetizacao_2023', 'idhm']:
    print(f'AUC-ROC bruta de {coluna} para atingiu_meta:', round(roc_auc_score(q['atingiu_meta'], q[coluna]), 3))
print('P(atingiu | folga>0) =', round(q.loc[q['folga_2023'] > 0, 'atingiu_meta'].mean(), 3), '| n =', int((q['folga_2023'] > 0).sum()))
print('P(atingiu | folga<=0) =', round(q.loc[q['folga_2023'] <= 0, 'atingiu_meta'].mean(), 3), '| n =', int((q['folga_2023'] <= 0).sum()))

linhas: 5222
corr(meta 2024, taxa 2023) = 0.979
corr(taxa 2024, taxa 2023) = 0.635
AUC-ROC bruta de folga_2023 para atingiu_meta: 0.487
AUC-ROC bruta de taxa_alfabetizacao_2023 para atingiu_meta: 0.484
AUC-ROC bruta de idhm para atingiu_meta: 0.503
P(atingiu | folga>0) = 0.55 | n = 909
P(atingiu | folga<=0) = 0.529 | n = 4313


Nenhuma variável isolada discrimina `atingiu_meta` (AUC-ROC entre 0,48 e 0,50). A meta de 2024 acompanha a taxa de 2023 (correlação 0,98) e a taxa de 2024 só se correlaciona 0,64 com a de 2023. Isso é coerente com o AUC-ROC de 0,570 da Fase 2 (README da Fase 2, linha 68) e antecipa que T2 pode ter pouco poder preditivo.

## 7. Escolas, municípios e tamanho de amostra

In [10]:
por_municipio = alunos[alunos['ano'] == 2024].groupby('id_municipio').size()
print('escolas distintas:', alunos['id_escola'].nunique())
print('alunos por município em 2024 (mediana, mínimo):', int(por_municipio.median()), int(por_municipio.min()))
print('municípios com alunos nos dois anos:', len(set(alunos.loc[alunos['ano'] == 2023, 'id_municipio']) & set(alunos.loc[alunos['ano'] == 2024, 'id_municipio'])))

escolas distintas: 42811
alunos por município em 2024 (mediana, mínimo): 122 10


municípios com alunos nos dois anos: 4844


Cada município tem em mediana 122 alunos em 2024 (mínimo 10), então a taxa municipal é estimada de poucos alunos e é ruidosa. Os alunos de um município compartilham os mesmos atributos municipais, o que exige separar treino e teste por município.

## 8. Hipóteses e decisões de modelagem

| # | Pergunta de negócio | Hipótese | Evidência desta EDA | Decisão de modelagem |
|---|---|---|---|---|
| H1 | Quais fatores mais impactam a alfabetização? | Região e IDHM se associam à alfabetização, com efeito modesto | Norte 50,9% a Sul 64,5%; quintis de IDHM de 55,1% a 60,8% | Usar região e IDHM como atributos de T1 e esperar poder preditivo limitado; comparar com o baseline e confirmar com SHAP |
| H2 | Quais municípios têm maior risco educacional? | O risco se concentra no Norte e no Nordeste | Menores proporções de alfabetizados na região Norte e no Nordeste | Ranking por probabilidade média prevista fora da amostra, conferida contra a taxa observada |
| H3 | Quais regiões têm padrões semelhantes? | Municípios se agrupam por perfil socioeconômico e o IDHM e seus componentes são redundantes | IDHM e componentes com correlação de 0,71 a 0,95 | Padronizar, aplicar PCA antes do K-Means e escolher K por cotovelo e silhouette |
| H4 | Quais municípios podem não atingir metas futuras? | O poder preditivo é baixo, porque a meta acompanha a taxa anterior e a taxa muda muito de um ano para o outro | Meta 2024 vs taxa 2023: 0,98; taxa 2024 vs taxa 2023: 0,64; AUC-ROC brutas de 0,48 a 0,50 | Decidir se a meta (conhecida de antemão) e a folga defasada entram como atributos; reportar a limitação e comparar com o `DummyClassifier` |
| H5 | Quais variáveis têm maior influência? | A importância se dilui entre variáveis colineares | IDHM e componentes com correlação de 0,71 a 0,95 | Interpretar SHAP por grupo de variáveis e checar a Permutation Importance |

As hipóteses são testadas nas unidades de modelagem e de interpretabilidade.

## 9. Perguntas que ficam para a unidade de pré-processamento

1. A população do modelo do aluno inclui os 13,3% que não fizeram a prova?
2. `meta_indicador` do ano seguinte e a folga defasada entram como atributos de T2?
3. Qual critério de imputação para o IDHM ausente de 6 municípios e para as `proporcao_aluno_nivel_*`?
4. Como separar treino, validação e teste por município mantendo a estratificação do alvo?